# Parameter optimization against cold-room experiments

This notebook calibrates SSA-evolution model parameters against laboratory measurements. It prepares isothermal and temperature-gradient experiments, evaluates the fitted model trajectories, reports error metrics, and plots selected experiment comparisons.

## Inputs and workflow
- Reads experiment data from `ssa_bibio.xlsx` and `SSA2014-2018_Hachikubo.xls`.
- Interpolates measurements onto hourly time series, including extended-duration cases used for selected plots.
- Fits isothermal parameters first, then evaluates temperature-gradient behavior and candidate B25 parameterizations.
- Reports RMSE in SSA and optical-diameter units and visualizes example trajectories.

Run cells from top to bottom so that experiment dictionaries and fitted parameter values are defined before the evaluation and plotting cells. The spreadsheets must be available in the working directory, unless their paths are changed in the loading cells.

In [ ]:
# %matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import function_SSA as core_ssa

In [ ]:
sphericity_dict = {'PP' : 0.5, 'DF' : 0.5, 'RG' : 0.99, 'FC' : 0, 'DH' : 0, 'MF' : 0.99, 'IF' : 0.5, 'SH' : 0.5, 'PPgp' : 0.5}

def evolve_do(df_layer, dt, og = False, exponant = 0.4):
    #B21
    #dt in second
    do_toy = []
    for index, row in df_layer.reset_index().iterrows():
        if index == 0:
            s_0 = sphericity_dict[row.grain_type]
            d_0 = 6/(row.ssa*917)
            do_toy.append(d_0)
        elif index == 1 :
            s_f = np.clip(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant) * dt * 3600 + s_0, a_min = 0, a_max = 1)
            do = core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, d_0, og, exponant = exponant)* dt * 3600 + d_0
            do_toy.append(do)
        else:
            # print(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant))
            s_f = np.clip(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant)* dt * 3600 + s_f, a_min = 0, a_max = 1)
            # print(core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, do, og, exponant = exponant))
            do = core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, do, og, exponant = exponant)* dt * 3600 + do
            do_toy.append(do)

    return np.array(do_toy)


def evolve_ssa_B25(df_layer, dt, a_A = 1320, a_B = 1.52e-2, a_L = 1.09e-2, ssa_minimum = 0, a_tan = 0.3):
    #B25
    #dt in hour
    ssa_b25 = []
    for index, row in df_layer.reset_index().iterrows():
        if index == 0:
                ssa = row.ssa
                ssa_b25.append(row.ssa)
        else:
            ssa = core_ssa.ssa_B25(ssa, row.temperature + 273.15, row.gradient, row.density, dt, 
                                    a_A = a_A, 
                                    a_B = a_B, 
                                    a_L = a_L, 
                                    ssa_minimum = ssa_minimum, 
                                    a_tan = a_tan)
            ssa_b25.append(ssa)

    return np.array(ssa_b25)


def ssa_2_do(ssa):
    return 6/(916.7*ssa)

def compute_rmse_experiment(experiment, model):
    l_do_rmse, l_do_bias = [], []
    l_ssa_rmse, l_ssa_bias = [], []
    for key, df in experiment.items():
        # if df.ssa.values[0] > 18:
        rmse = np.sqrt(np.mean((ssa_2_do(df[model]) - ssa_2_do(df.ssa))**2))
        bias = np.mean(ssa_2_do(df.ssa) -ssa_2_do(df[model]))
        l_do_rmse.append(rmse)
        l_do_bias.append(bias)

        rmse = np.sqrt(np.mean(df[model] - df.ssa)**2)
        bias = np.mean(df.ssa - df[model])
        l_ssa_rmse.append(rmse)
        l_ssa_bias.append(bias)

    return np.mean(l_do_rmse), np.mean(l_do_bias), np.mean(l_ssa_rmse), np.mean(l_ssa_bias)


def interpolate_experiment(df, dt = 1, max_days = 200, extend = False):
    # Interpolate the experiment to a specific dt
    # dt in hour
    density = df['density (kg/m3)']
    ssa = df['ssa']
    time = df['time'].values
    if extend:
        time_hourly = np.arange(0, max_days*24 + dt , dt)
    else:
        if time[-1] > max_days*24:
            time_hourly = np.arange(0, max_days*24 + dt , dt)
        else:
            time_hourly = np.arange(0, time[-1] + dt , dt)
    density_hourly = np.interp(time_hourly, time, density)
    ssa_hourly = np.interp(time_hourly, time, ssa)

    df_hourly = pd.DataFrame({'time' : time_hourly, 
                              'gradient' : df['tempe gradient'].values[0], 
                              'temperature' : df['tempe'].values[0],
                              'density' : density_hourly,
                              'ssa' : ssa_hourly,
                              'grain_type' : df['snow type'].values[0]})

    return df_hourly

def interpolate_experiment_japan(df, dt = 1, max_days = 200, extend = False):
    # Interpolate the experiment to a specific dt
    # dt in hour
    ssa = np.array(df.ssa, dtype = float)
    time = np.array(df.time, dtype = float)
    if extend:
        time_hourly = np.arange(0, max_days*24 + dt , dt)
    else:
        if time[-1] > max_days*24:
            time_hourly = np.arange(0, max_days*24 + dt , dt)
        else:
            time_hourly = np.arange(0, time[-1] + dt , dt)

    ssa_hourly = np.interp(time_hourly, time, ssa)

    df_hourly = pd.DataFrame({'time' : time_hourly,
                              'density' : 50,
                              'gradient' : df.gradient.values[0], 
                              'temperature' : df.temperature.values[0] - 273.15,
                              'ssa' : ssa_hourly,
                              'grain_type' : 'PP'})

    return df_hourly



### gradient

In [ ]:
name = "gradient"
file = 'ssa_bibio.xlsx'
df = pd.read_excel(file, sheet_name=name, header=0)

experiment = {}
for name, group in df.groupby(df.exp):
    if ('Wa' not in name) and ('B1' not in name):
        group.rename(columns = {'time (h)' : 'time', 'ssa (m2/kg)' : 'ssa'}, inplace=True)
        experiment[name] = group
        # experiment[name]['dt'] = experiment[name]['time'].diff()
        # experiment[name]['dt'].iloc[0] = experiment[name]['time'].iloc[0]
        # experiment[name]


experiment_TG = {}
for key, df in experiment.items():
    if df.ssa.values[0] > 18:
        experiment_TG[key] = df

In [ ]:
experiment_TG_hourly = {}
for name, df in experiment_TG.items():
    experiment_TG_hourly[name] = interpolate_experiment(df, dt = 1)

In [ ]:
experiment_TG_hourly_75 = {}
for name, df in experiment_TG.items():
    experiment_TG_hourly_75[name] = interpolate_experiment(df, dt = 1, max_days = 90, extend = True)


Isothermal

In [ ]:
name = "iso"
file = 'ssa_bibio.xlsx'
df = pd.read_excel(file, sheet_name=name, header=0)

experiment_iso = {}
for name, group in df.groupby(df.exp):
    group.rename(columns = {'time (h)' : 'time', 'ssa (m2/kg)' : 'ssa'}, inplace=True)
    experiment_iso[name] = group
    # experiment_iso[name]['dt'] = experiment_iso[name]['time'].diff()
    # experiment_iso[name]['dt'].iloc[0] = experiment_iso[name]['time'].iloc[0]

# for key, df in experiment_iso.items():
#     df.rename(columns = {'time (h)' : 'time', 'ssa (m2/kg)' : 'ssa'}, inplace=True)

In [ ]:
experiment_iso

In [ ]:
df_japan_2013 = pd.read_excel('SSA2014-2018_Hachikubo.xls', sheet_name = 'main2013')

five_multiple = [i for i in range(len(df_japan_2013.columns)+1)][::5]

experiment_japan_2013 = {}
count = 1
for i in range(len(five_multiple)):
    if i < len(five_multiple) - 1:
        df = df_japan_2013.iloc[:,five_multiple[i]:five_multiple[i+1]]
        name = 'H' + str(count) +'-13'
        temperature = float(df.iloc[0,0][:3])
        columns = df.columns[0]
        df_clean = df.iloc[3:,:].reset_index()
        time = df_clean.iloc[:,2].values - df_clean.iloc[:,2].values[0]
        experiment_japan_2013[name] = pd.DataFrame({'time': time, 'ssa' :  df_clean.iloc[:,3], 'temperature' : temperature, 'gradient' :  0}).dropna()
        count += 1

df_japan_2014 = pd.read_excel('SSA2014-2018_Hachikubo.xls', sheet_name = 'main2014')

five_multiple = [i for i in range(len(df_japan_2014.columns)+1)][::5]

experiment_japan_2014 = {}
count = 1
for i in range(len(five_multiple)):
    if i < len(five_multiple) - 1:
        df = df_japan_2014.iloc[:,five_multiple[i]:five_multiple[i+1]]
        name = df.columns[0]
        if ('B' in name) or ('E' in name) or 'F' in name:
            continue
        else:
            name = 'H' + str(count) +'-14'
            temperature = float(df.iloc[0,0][:3])
            columns = df.columns[0]
            df_clean = df.iloc[3:,:].reset_index()
            time = df_clean.iloc[:,2].values - df_clean.iloc[:,2].values[0]
            experiment_japan_2014[name] = pd.DataFrame({'time': time, 'ssa' :  df_clean.iloc[:,3], 'temperature' : temperature, 'gradient' :  0}).dropna()
            count += 1

In [ ]:
experiment_iso_hourly = {}
for name, df in experiment_iso.items():
    experiment_iso_hourly[name] = interpolate_experiment(df, dt = 1, max_days = 200)

experiment_japan_2013_hourly = {}
for key, df in experiment_japan_2013.items():
    experiment_japan_2013_hourly[key] = interpolate_experiment_japan(df, max_days = 200)

experiment_japan_2014_hourly = {}
for key, df in experiment_japan_2014.items():
    experiment_japan_2014_hourly[key] = interpolate_experiment_japan(df, max_days = 200)

experiment_iso_hourly.update(experiment_japan_2013_hourly)
experiment_iso_hourly.update(experiment_japan_2014_hourly)

In [ ]:
experiment_japan_2014_hourly

In [ ]:
experiment_iso_hourly_200 = {}
for name, df in experiment_iso.items():
    experiment_iso_hourly_200[name] = interpolate_experiment(df, dt = 1, max_days = 200, extend = True)

experiment_japan_2013_hourly_200 = {}
for key, df in experiment_japan_2013.items():
    experiment_japan_2013_hourly_200[key] = interpolate_experiment_japan(df, max_days = 200, extend = True)

experiment_japan_2014_hourly_200 = {}
for key, df in experiment_japan_2014.items():
    experiment_japan_2014_hourly_200[key] = interpolate_experiment_japan(df, max_days = 200, extend = True)

experiment_iso_hourly_200.update(experiment_japan_2013_hourly_200)
experiment_iso_hourly_200.update(experiment_japan_2014_hourly_200)

In [ ]:
experiment_iso.update(experiment_japan_2013)
experiment_iso.update(experiment_japan_2014)

## Isothermal

### Optimization under isothermal

In [ ]:
popt_iso_B = [0.11444]
# popt_iso = [0.087033, 0.13861171]
popt_iso = [0.082378, 906.580521]


In [ ]:
dt = 1
for key, df in experiment_iso_hourly.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso_B[0])
    df['ssa_model_B25_opt_B'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso[0], a_L = popt_iso[1])
    df['ssa_model_B25_opt'] = ssa_b25


ET_rmse_do_b21, ET_bias_do_b21, ET_rmse_ssa_b21, ET_bias_ssa_b21 = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B21')
ET_rmse_do_b25, ET_bias_do_b25, ET_rmse_ssa_b25, ET_bias_ssa_b25 = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25')
ET_rmse_do_b25_b, ET_bias_do_b25_b, ET_rmse_ssa_b25_b, ET_bias_ssa_b25_b = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25_opt')


In [ ]:
print('RMSE')
print(f"RMSE B21 (m2/kg) : {ET_rmse_ssa_b21}")
print(f"RMSE B25 param ssa/rho(m2/kg) : {ET_rmse_ssa_b25}")
print(f"RMSE B25 param ssa/rho(m2/kg) opt b and l  : {ET_rmse_ssa_b25_b}")

print('Bias')
print(f"bias B21 (m2/kg) : {ET_bias_ssa_b21}")
print(f"bias B25 param ssa/rho(m2/kg) : {ET_bias_ssa_b25}")
print(f"bias B25 param ssa/rho(m2/kg) opt b an l  : {ET_bias_ssa_b25_b}")

In [ ]:
print('RMSE')
print(f"RMSE B21 (mm) : {ET_rmse_do_b21*1e3}")
print(f"RMSE B25 param ssa/rho(mm) : {ET_rmse_do_b25*1e3}")
print(f"RMSE B25 param ssa/rho(mm) opt b and l  : {ET_rmse_do_b25_b*1e3}")

print('Bias')
print(f"bias B21 (mm) : {ET_bias_do_b21*1e3}")
print(f"bias B25 param ssa/rho(mm) : {ET_bias_do_b25*1e3}")
print(f"bias B25 param ssa/rho(mm) opt b an l  : {ET_bias_do_b25_b*1e3}")


In [ ]:
#Recalculate for graph at 200 days

for key, df in experiment_iso_hourly_200.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso_B[0])
    df['ssa_model_B25_opt_B'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso[0], a_L = popt_iso[1])
    df['ssa_model_B25_opt'] = ssa_b25


In [ ]:
fig, ax = plt.subplots(5,5, figsize = (9,8), sharey = True, sharex = True)
tab20 = plt.get_cmap('tab20').colors
count = 0
ax_f = ax.flatten()
for key, df in experiment_iso_hourly_200.items():
    ax_f[count].set_title(f"{key}, T = {np.round(df.temperature.values[0],1)}$\degree$C", size = 10)
    ax_f[count].plot(df.time/24, df.ssa_model_B21, alpha = 0.7, label = 'B21')
    ax_f[count].plot(df.time/24, df.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
    ax_f[count].scatter(experiment_iso[key]['time']/24, experiment_iso[key]['ssa'], alpha = 0.5, color = 'k')
    count += 1
    ax_f[count].grid(True, linestyle="--", alpha=0.5)

ax_f[0].legend(fontsize = 9)
plt.xlim(-5,200)
# plt.subplots_adjust(hspace = 0.45)
ax_f[0].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[5].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[10].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[15].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[20].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[20].set_xlabel('Time (days)')
ax_f[21].set_xlabel('Time (days)')
ax_f[22].set_xlabel('Time (days)')
ax_f.flat[-1].set_visible(False)
ax_f.flat[-2].set_visible(False)
fig.tight_layout()

# fig.savefig('/home/jum002/public_html/ssa/cold_room_isothermal_all.png', dpi = 300)

In [ ]:
f1 = experiment_iso_hourly_200['F1-03']
h1 = experiment_iso_hourly_200['H1-14']
h2 = experiment_iso_hourly_200['H2-14']

fig, ax = plt.subplots(1,3, figsize = (8,3), sharey = True, sharex = True)
ax_f = ax.flatten()
ax_f[0].set_title(f"{'F1-03'}, $T$ = {np.round(f1.temperature.values[0],1)} $C\degree$, {f1.grain_type[0]}", size = 10)
ax_f[0].plot(f1.time/24, f1.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[0].plot(f1.time/24, f1.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[0].scatter(experiment_iso['F1-03']['time']/24, experiment_iso['F1-03']['ssa'], alpha = 0.5, color = 'k')

ax_f[1].set_title(f"{'H1-14'}, $T$ = {np.round(h1.temperature.values[0],1)} $C\degree$, {h1.grain_type[0]}", size = 10)
ax_f[1].plot(h1.time/24, h1.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[1].plot(h1.time/24, h1.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[1].scatter(experiment_iso['H1-14']['time']/24, experiment_iso['H1-14']['ssa'], alpha = 0.5, color = 'k')

ax_f[2].set_title(f"{'H2-04'}, $T$ = {np.round(h2.temperature.values[0],1)} $C\degree$, {h2.grain_type[0]}", size = 10)
ax_f[2].plot(h2.time/24, h2.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[2].plot(h2.time/24, h2.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[2].scatter(experiment_iso['H2-14']['time']/24, experiment_iso['H2-14']['ssa'], alpha = 0.5, color = 'k')


ax_f[0].legend(fontsize = 9)
plt.xlim(-5,200)
# plt.subplots_adjust(hspace = 0.45)
ax_f[0].set_ylabel('SSA ($m^2 kg^{-1}$)')
ax_f[0].set_xlabel('Time (days)')
ax_f[1].set_xlabel('Time (days)')
ax_f[2].set_xlabel('Time (days)')
fig.tight_layout()

# fig.savefig('/home/jum002/public_html/ssa/cold_room_isothermal_subset.png', dpi = 300)

### Optimization under TG

In [ ]:
dt = 1
def function_b25_optimize(experiment, a_A):
    ssa = np.concat([evolve_ssa_B25(df, dt, a_A = a_A, a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 0) for key, df in experiment.items()])
    #concat both metric (ssa and do) and normalize
    return np.concat([ssa/np.max(ssa), ssa_2_do(ssa)/np.max(ssa_2_do(ssa))])


y_ssa = np.concatenate([df.ssa.values for key, df in experiment_TG_hourly.items()])
y_do = np.concatenate([ssa_2_do(df.ssa.values) for key, df in experiment_TG_hourly.items()])
y_data = np.concatenate([y_ssa/np.max(y_ssa), y_do/np.max(y_do)])
lower = [100]
upper = [1e5]
p0 = [1000]
popt_TG_A, pcov = curve_fit(function_b25_optimize, experiment_TG_hourly, y_data)#, p0 = p0, bounds = (lower, upper))
popt_TG_A

In [ ]:
dt = 1
def function_b25_optimize(experiment, a_A):
    ssa = np.concat([evolve_ssa_B25(df, dt, a_A = a_A, a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 8) for key, df in experiment.items()])
    # return ssa_2_do(ssa)
    return ssa

# y_data = np.concatenate([ssa_2_do(df.ssa.values) for key, df in experiment_TG_hourly.items()])
y_data = np.concatenate([df.ssa.values for key, df in experiment_TG_hourly.items()])

lower = [100]
upper = [1e5]
p0 = [1000]
popt_TG_A_min, pcov = curve_fit(function_b25_optimize, experiment_TG_hourly, y_data)#, p0 = p0, bounds = (lower, upper))
popt_TG_A_min

In [ ]:
dt = 1
def function_b25_optimize(experiment, a_A):
    ssa = np.concat([evolve_ssa_B25(df, dt, a_A = a_A, a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 8) for key, df in experiment.items()])
    #concat both metric (ssa and do) and normalize
    return np.concat([ssa/np.max(ssa), ssa_2_do(ssa)/np.max(ssa_2_do(ssa))])

y_ssa = np.concatenate([df.ssa.values for key, df in experiment_TG_hourly.items()])
y_do = np.concatenate([ssa_2_do(df.ssa.values) for key, df in experiment_TG_hourly.items()])
y_data = np.concatenate([y_ssa/np.max(y_ssa), y_do/np.max(y_do)])
lower = [100]
upper = [1e5]
p0 = [1000]
popt_TG_A_min, pcov = curve_fit(function_b25_optimize, experiment_TG_hourly, y_data)#, p0 = p0, bounds = (lower, upper))
popt_TG_A_min

In [ ]:
dt = 1
for key, df in experiment_TG_hourly.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_A = popt_TG_A[0], a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 0)
    df['ssa_model_B25_opt_A'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_A = popt_TG_A_min[0], a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 8)
    df['ssa_model_B25_opt_A_min'] = ssa_b25


TG_rmse_do_b21, TG_bias_do_b21, TG_rmse_ssa_b21, TG_bias_ssa_b21 = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B21')
TG_rmse_do_b25, TG_bias_do_b25, TG_rmse_ssa_b25, TG_bias_ssa_b25 = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25')
TG_rmse_do_b25_A, TG_bias_do_b25_A, TG_rmse_ssa_b25_A, TG_bias_ssa_b25_A = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25_opt_A')
TG_rmse_do_b25_A_min, TG_bias_do_b25_A_min, TG_rmse_ssa_b25_A_min, TG_bias_ssa_b25_A_min = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25_opt_A_min')

In [ ]:
print(f"RMSE B21 (m2/kg) : {TG_rmse_ssa_b21}")
print(f"RMSE B25  opt a_A (m2/kg) : {TG_rmse_ssa_b25_A}")
print(f"RMSE B25 a_A, min (m2/kg) : {TG_rmse_ssa_b25_A_min}")

#bias
print(f"bias B21 (m2/kg) : {TG_bias_ssa_b21}")
print(f"bias B25  opt a_A (m2/kg) : {TG_bias_ssa_b25_A}")
print(f"bias B25  opt a_A, min (m2/kg) : {TG_bias_ssa_b25_A_min}")

In [ ]:
print(f"RMSE B21 (mm) : {TG_rmse_do_b21*1e3}")
print(f"RMSE B25  opt a_A (mm) : {TG_rmse_do_b25_A*1e3}")
print(f"RMSE B25 a_A, min (mm) : {TG_rmse_do_b25_A_min*1e3}")

#bias
print(f"bias B21 (mm) : {TG_bias_do_b21*1e3}")
print(f"bias B25  opt a_A (mm) : {TG_bias_do_b25_A*1e3}")
print(f"bias B25  opt a_A, min (mm) : {TG_bias_do_b25_A_min*1e3}")

In [ ]:
print(f"RMSE B21 (mm) : {np.mean([TG_rmse_do_b21*1e3, ET_rmse_do_b21*1e3])}")
print(f"RMSE B25 a_A, min (mm) : {np.mean([TG_rmse_do_b25_A_min*1e3, ET_rmse_do_b25_b*1e3])}")

#bias
print(f"bias B21 (mm) : {np.mean([TG_bias_do_b21*1e3, ET_bias_do_b21*1e3])}")
print(f"bias B25  opt a_A, min (mm) : {np.mean([TG_bias_do_b25_A_min*1e3, ET_bias_do_b25_b*1e3])}")

In [ ]:
print(f"RMSE B21 (m2/kg) : {np.mean([TG_rmse_ssa_b21, ET_rmse_ssa_b21])}")
print(f"RMSE B25 a_A, min (m2/kg) : {np.mean([TG_rmse_ssa_b25_A_min, ET_rmse_ssa_b25_b])}")

#bias
print(f"bias B21 (m2/kg) : {np.mean([TG_bias_ssa_b21, ET_bias_ssa_b21])}")
print(f"bias B25  opt a_A, min (m2/kg) : {np.mean([TG_bias_ssa_b25_A_min, ET_bias_ssa_b25_b])}")

In [ ]:
dt = 1
for key, df in experiment_TG_hourly_75.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_A = popt_TG_A[0], a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 0)
    df['ssa_model_B25_opt_A'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_A = popt_TG_A_min[0], a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 8)
    df['ssa_model_B25_opt_A_min'] = ssa_b25

### Temperature gradient

In [ ]:
fig, ax = plt.subplots(3,3, figsize = (7,7), sharey = True, sharex = True)
tab20 = plt.get_cmap('tab20').colors
count = 0
ax_f = ax.flatten()
for key, df in experiment_TG_hourly_75.items():
    if df.ssa.values[0] > 18:
        ax_f[count].set_title(f"{key}, $\\nabla$T = {df.gradient.values[0]} K m$^{{-1}}$, T = {df.temperature.values[0]}$\degree$C", fontsize = 8)
        ax_f[count].plot(df.time/24, df.ssa_model_B21, alpha = 0.7, label = 'B21')
        ax_f[count].plot(df.time/24, df.ssa_model_B25_opt_A_min, alpha = 0.7, label = 'B25')
        ax_f[count].scatter(experiment[key]['time']/24, experiment[key]['ssa'], alpha = 0.5, color = 'k')
        ax_f[count].grid(True, linestyle="--", alpha=0.5)
        count += 1

plt.legend(fontsize = 'small')
# plt.ylim(5,45)
# plt.subplots_adjust(hspace = 0.25)
ax_f[0].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[3].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[6].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[6].set_xlabel('Time (days)')
ax_f[7].set_xlabel('Time (days)')
ax_f[8].set_xlabel('Time (days)')
fig.tight_layout()

# fig.savefig('/home/jum002/public_html/ssa/cold_room_gradient.png', dpi = 300)

In [ ]:
b2 = experiment_TG_hourly_75['B2-23']
b3 = experiment_TG_hourly_75['B3-23']
c1 = experiment_TG_hourly_75['C1-14']

In [ ]:
fig, ax = plt.subplots(2,3, figsize = (8,5), sharey = 'row', sharex = True)

ax_f = ax.flatten()
#ET
ax_f[0].set_title(f"{'F1-03'}, $\\nabla$T = 0 K m$^{{-1}}$, T = {np.round(f1.temperature.values[0],1)}$\degree$C", size = 9)
ax_f[0].plot(f1.time/24, f1.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[0].plot(f1.time/24, f1.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[0].scatter(experiment_iso['F1-03']['time']/24, experiment_iso['F1-03']['ssa'], alpha = 0.5, color = 'k')
ax_f[0].grid(True, linestyle="--", alpha=0.5)

ax_f[1].set_title(f"{'H1-14'}, $\\nabla$T = 0 K m$^{{-1}}$, T = {np.round(h1.temperature.values[0],1)}$\degree$C", size = 9)
ax_f[1].plot(h1.time/24, h1.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[1].plot(h1.time/24, h1.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[1].scatter(experiment_iso['H1-14']['time']/24, experiment_iso['H1-14']['ssa'], alpha = 0.5, color = 'k')
ax_f[1].grid(True, linestyle="--", alpha=0.5)

ax_f[2].set_title(f"{'H2-14'}, $\\nabla$T = 0 K m$^{{-1}}$, T = {np.round(h2.temperature.values[0],1)}$\degree$C", size = 9)
ax_f[2].plot(h2.time/24, h2.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[2].plot(h2.time/24, h2.ssa_model_B25_opt, alpha = 0.7, label = 'B25')
ax_f[2].scatter(experiment_iso['H2-14']['time']/24, experiment_iso['H2-14']['ssa'], alpha = 0.5, color = 'k')
ax_f[2].grid(True, linestyle="--", alpha=0.5)

#TG
ax_f[3].set_title(f"{'C1-14'}, $\\nabla$T = {c1.gradient.values[0]} K m$^{{-1}}$, T = {c1.temperature.values[0]}$\degree$C", fontsize = 9)
ax_f[3].plot(c1.time/24, c1.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[3].plot(c1.time/24, c1.ssa_model_B25_opt_A_min, alpha = 0.7, label = 'B25')
ax_f[3].scatter(experiment['C1-14']['time']/24, experiment['C1-14']['ssa'], alpha = 0.5, color = 'k')
ax_f[3].grid(True, linestyle="--", alpha=0.5)

ax_f[4].set_title(f"{'B2-23'}, $\\nabla$T = {b2.gradient.values[0]} K m$^{{-1}}$, T = {b2.temperature.values[0]}$\degree$C", fontsize = 9)
ax_f[4].plot(b2.time/24, b2.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[4].plot(b2.time/24, b2.ssa_model_B25_opt_A_min, alpha = 0.7, label = 'B25')
ax_f[4].scatter(experiment['B2-23']['time']/24, experiment['B2-23']['ssa'], alpha = 0.5, color = 'k')
ax_f[4].grid(True, linestyle="--", alpha=0.5)

ax_f[5].set_title(f"{'B3-23'}, $\\nabla$T = {b3.gradient.values[0]} K m$^{{-1}}$, T = {b3.temperature.values[0]}$\degree$C", fontsize = 9)
ax_f[5].plot(b3.time/24, b3.ssa_model_B21, alpha = 0.7, label = 'B21')
ax_f[5].plot(b3.time/24, b3.ssa_model_B25_opt_A_min, alpha = 0.7, label = 'B25')
ax_f[5].scatter(experiment['B3-23']['time']/24, experiment['B3-23']['ssa'], alpha = 0.5, color = 'k')
ax_f[5].grid(True, linestyle="--", alpha=0.5)

plt.legend(fontsize = 'small')
plt.xlim(-1,30)
# plt.subplots_adjust(hspace = 0.25)
ax_f[0].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[3].set_ylabel('SSA (m$^2$ kg$^{-1}$)')
ax_f[3].set_xlabel('Time (days)')
ax_f[4].set_xlabel('Time (days)')
ax_f[5].set_xlabel('Time (days)')
fig.text(0.53, 0.98, 'ET', ha='center', fontsize=14, fontweight='bold')
fig.text(0.53, 0.52, 'TG', ha='center', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 1])
fig.subplots_adjust(hspace = 0.25)
# fig.savefig('/home/jum002/public_html/ssa/cold_room_subset.png', dpi = 300, bbox_inches='tight')